## Preprocessing

In [1]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import tensorflow as tf


# read in the cleaned csv file from online site (data stored on private server to provide stable static hosting)
# df = pd.read_csv("http://www.andrewlane.us/data/crime_data2020-2024.csv") # File is 268MB, allow time for download
df = pd.read_csv("resources/cleaned_data/crime_2020.csv") # Small file for quick testing
df.head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON,crime_timestamp,Year
0,190326475,03/01/2020 12:00:00 AM,2020-03-01,2130,7,Wilshire,784,1,510,VEHICLE - STOLEN,...,510.0,998.0,NaN,NaN,1900 S LONGWOOD AV,NaN,34.0375,-118.3506,2020-03-01 21:30:00,2020
1,200106753,02/09/2020 12:00:00 AM,2020-02-08,1800,1,Central,182,1,330,BURGLARY FROM VEHICLE,...,330.0,998.0,NaN,NaN,1000 S FLOWER ST,NaN,34.0444,-118.2628,2020-02-08 18:00:00,2020
2,200320258,11/11/2020 12:00:00 AM,2020-11-04,1700,3,Southwest,356,1,480,BIKE - STOLEN,...,480.0,NaN,NaN,NaN,1400 W 37TH ST,NaN,34.0210,-118.3002,2020-11-04 17:00:00,2020
3,200907217,05/10/2023 12:00:00 AM,2020-03-10,2037,9,Van Nuys,964,1,343,SHOPLIFTING-GRAND THEFT ($950.01 & OVER),...,343.0,NaN,NaN,NaN,14000 RIVERSIDE DR,NaN,34.1576,-118.4387,2020-03-10 20:37:00,2020
4,200412582,09/09/2020 12:00:00 AM,2020-09-09,630,4,Hollenbeck,413,1,510,VEHICLE - STOLEN,...,510.0,NaN,NaN,NaN,200 E AVENUE 28,NaN,34.0820,-118.2130,2020-09-09 06:30:00,2020


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199805 entries, 0 to 199804
Data columns (total 30 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   DR_NO            199805 non-null  int64  
 1   Date Rptd        199805 non-null  object 
 2   DATE OCC         199805 non-null  object 
 3   TIME OCC         199805 non-null  int64  
 4   AREA             199805 non-null  int64  
 5   AREA NAME        199805 non-null  object 
 6   Rpt Dist No      199805 non-null  int64  
 7   Part 1-2         199805 non-null  int64  
 8   Crm Cd           199805 non-null  int64  
 9   Crm Cd Desc      199805 non-null  object 
 10  Mocodes          173050 non-null  object 
 11  Vict Age         199805 non-null  int64  
 12  Vict Sex         199805 non-null  object 
 13  Vict Descent     174316 non-null  object 
 14  Premis Cd        199803 non-null  float64
 15  Premis Desc      199736 non-null  object 
 16  Weapon Used Cd   72974 non-null   floa

In [3]:
# remove columns with serialized and descriptive values that are expressed in codes 
df = df.drop(columns=['DR_NO', 'Date Rptd', 'DATE OCC', 'AREA NAME', 'Crm Cd Desc', 
                      'Mocodes', 'Premis Desc', 'Weapon Desc', 
                      'Crm Cd 1', 'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 
                      'LOCATION', 'LAT', 'LON',
                      'Cross Street', 'crime_timestamp'])

In [4]:
codes = pd.read_csv("resources/crime_codes_final.csv")
df = pd.merge(df, codes, left_on='Crm Cd', right_on='crime_code')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199805 entries, 0 to 199804
Data columns (total 18 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   TIME OCC                   199805 non-null  int64  
 1   AREA                       199805 non-null  int64  
 2   Rpt Dist No                199805 non-null  int64  
 3   Part 1-2                   199805 non-null  int64  
 4   Crm Cd                     199805 non-null  int64  
 5   Vict Age                   199805 non-null  int64  
 6   Vict Sex                   199805 non-null  object 
 7   Vict Descent               174316 non-null  object 
 8   Premis Cd                  199803 non-null  float64
 9   Weapon Used Cd             72974 non-null   float64
 10  Status                     199805 non-null  object 
 11  Status Desc                199805 non-null  object 
 12  Year                       199805 non-null  int64  
 13  crime_code                 19

In [5]:
df = df.drop(columns=['crime_description', 'crime_subcategory', 'crime_subcategory_mapping'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199805 entries, 0 to 199804
Data columns (total 15 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   TIME OCC        199805 non-null  int64  
 1   AREA            199805 non-null  int64  
 2   Rpt Dist No     199805 non-null  int64  
 3   Part 1-2        199805 non-null  int64  
 4   Crm Cd          199805 non-null  int64  
 5   Vict Age        199805 non-null  int64  
 6   Vict Sex        199805 non-null  object 
 7   Vict Descent    174316 non-null  object 
 8   Premis Cd       199803 non-null  float64
 9   Weapon Used Cd  72974 non-null   float64
 10  Status          199805 non-null  object 
 11  Status Desc     199805 non-null  object 
 12  Year            199805 non-null  int64  
 13  crime_code      199805 non-null  int64  
 14  crime_category  199805 non-null  object 
dtypes: float64(2), int64(8), object(5)
memory usage: 22.9+ MB


In [6]:
df = df.loc[df['crime_category'] == 'Violent Crimes']
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 67981 entries, 58308 to 199792
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   TIME OCC        67981 non-null  int64  
 1   AREA            67981 non-null  int64  
 2   Rpt Dist No     67981 non-null  int64  
 3   Part 1-2        67981 non-null  int64  
 4   Crm Cd          67981 non-null  int64  
 5   Vict Age        67981 non-null  int64  
 6   Vict Sex        67981 non-null  object 
 7   Vict Descent    67962 non-null  object 
 8   Premis Cd       67981 non-null  float64
 9   Weapon Used Cd  64892 non-null  float64
 10  Status          67981 non-null  object 
 11  Status Desc     67981 non-null  object 
 12  Year            67981 non-null  int64  
 13  crime_code      67981 non-null  int64  
 14  crime_category  67981 non-null  object 
dtypes: float64(2), int64(8), object(5)
memory usage: 8.3+ MB


In [7]:
df['Status'].value_counts()

Status
IC    36275
AO    18503
AA    12411
JA      545
JO      247
Name: count, dtype: int64

In [8]:
df['Status Desc'].value_counts()

Status Desc
Invest Cont     36275
Adult Other     18503
Adult Arrest    12411
Juv Arrest        545
Juv Other         247
Name: count, dtype: int64

In [9]:
df.nunique()

TIME OCC          1234
AREA                21
Rpt Dist No       1138
Part 1-2             2
Crm Cd              46
Vict Age           100
Vict Sex             3
Vict Descent        19
Premis Cd          266
Weapon Used Cd      76
Status               5
Status Desc          5
Year                 1
crime_code          46
crime_category       1
dtype: int64

In [10]:
# create column Arrest to use to train model
df['Arrest'] = df['Status'].apply(lambda x: 1 if x == "AA" or x == "JA" else 0)

# drop the Vict Age column to keep it out of the training data
df = df.drop(columns=['Status', 'Status Desc', 'crime_category'])

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 67981 entries, 58308 to 199792
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   TIME OCC        67981 non-null  int64  
 1   AREA            67981 non-null  int64  
 2   Rpt Dist No     67981 non-null  int64  
 3   Part 1-2        67981 non-null  int64  
 4   Crm Cd          67981 non-null  int64  
 5   Vict Age        67981 non-null  int64  
 6   Vict Sex        67981 non-null  object 
 7   Vict Descent    67962 non-null  object 
 8   Premis Cd       67981 non-null  float64
 9   Weapon Used Cd  64892 non-null  float64
 10  Year            67981 non-null  int64  
 11  crime_code      67981 non-null  int64  
 12  Arrest          67981 non-null  int64  
dtypes: float64(2), int64(9), object(2)
memory usage: 7.3+ MB


In [12]:
# df['Mocodes'] = df['Mocodes'].fillna(0)
# df['Mocodes'] = df['Mocodes'].astype(int)

In [13]:
# Convert categorical data to numeric with `pd.get_dummies`
df = pd.get_dummies(df, columns=['Vict Sex', 'Vict Descent'])
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 67981 entries, 58308 to 199792
Data columns (total 33 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   TIME OCC          67981 non-null  int64  
 1   AREA              67981 non-null  int64  
 2   Rpt Dist No       67981 non-null  int64  
 3   Part 1-2          67981 non-null  int64  
 4   Crm Cd            67981 non-null  int64  
 5   Vict Age          67981 non-null  int64  
 6   Premis Cd         67981 non-null  float64
 7   Weapon Used Cd    64892 non-null  float64
 8   Year              67981 non-null  int64  
 9   crime_code        67981 non-null  int64  
 10  Arrest            67981 non-null  int64  
 11  Vict Sex_F        67981 non-null  bool   
 12  Vict Sex_M        67981 non-null  bool   
 13  Vict Sex_Unknown  67981 non-null  bool   
 14  Vict Descent_A    67981 non-null  bool   
 15  Vict Descent_B    67981 non-null  bool   
 16  Vict Descent_C    67981 non-null  bool  

In [14]:
# Split our preprocessed data into our features and target arrays
y = df['Arrest'].values
X = df.drop('Arrest', axis=1).values

# Split the preprocessed data into a training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=8)


In [15]:
# Create a StandardScaler instances
scaler = StandardScaler()

# Fit the StandardScaler
X_scaler = scaler.fit(X_train)

# Scale the data
X_train = X_scaler.transform(X_train)
X_test = X_scaler.transform(X_test)

# Run Keras Tuner

In [16]:
# Create a method that creates a new Sequential model with hyperparameter options
def create_model(hp):
    nn_model = tf.keras.models.Sequential()

    # Allow kerastuner to decide which activation function to use in hidden layers
    activation = hp.Choice('activation',['relu','tanh'])

    # Allow kerastuner to decide number of neurons in first layer
    nn_model.add(tf.keras.layers.Dense(units=hp.Int('first_units',
        min_value=1,
        max_value=30,
        step=5), activation=activation, input_dim=37))

    # Allow kerastuner to decide number of hidden layers and neurons in hidden layers
    for i in range(hp.Int('num_layers', 1, 5)):
        nn_model.add(tf.keras.layers.Dense(units=hp.Int('units_' + str(i),
            min_value=1,
            max_value=30,
            step=5),
            activation=activation))

    nn_model.add(tf.keras.layers.Dense(units=1, activation="sigmoid"))

    # Compile the model
    nn_model.compile(loss="binary_crossentropy", optimizer='adam', metrics=["accuracy"])

    return nn_model

In [17]:
# Import the kerastuner library
import keras_tuner as kt

tuner = kt.Hyperband(
    create_model,
    objective="val_accuracy",
    max_epochs=20,
    hyperband_iterations=2)

X_train

Reloading Tuner from ./untitled_project/tuner0.json


array([[-0.25154409,  0.53404285,  0.47252634, ...,  2.17558289,
        -0.19019775, -0.0088578 ],
       [ 0.0948914 ,  1.01837195,  0.99040805, ..., -0.45964693,
        -0.19019775, -0.0088578 ],
       [-1.8632222 ,  1.17981498,  1.14577256, ..., -0.45964693,
        -0.19019775, -0.0088578 ],
       ...,
       [ 0.07982898,  0.37259982,  0.34952943, ..., -0.45964693,
        -0.19019775, -0.0088578 ],
       [ 0.07229778, -1.24183052, -1.30121853, ..., -0.45964693,
        -0.19019775, -0.0088578 ],
       [ 0.3509524 ,  0.37259982,  0.42073817, ..., -0.45964693,
        -0.19019775, -0.0088578 ]])

In [18]:
# Run the kerastuner search for best hyperparameters
tuner.search(X_train,y_train,epochs=20,validation_data=(X_test,y_test))

In [19]:
top_hyper = tuner.get_best_hyperparameters(3)
for param in top_hyper:
    print(param.values)

{'activation': 'relu', 'first_units': 1, 'num_layers': 4, 'units_0': 1, 'units_1': 1, 'units_2': 26, 'units_3': 16, 'tuner/epochs': 3, 'tuner/initial_epoch': 0, 'tuner/bracket': 2, 'tuner/round': 0}
{'activation': 'relu', 'first_units': 6, 'num_layers': 4, 'units_0': 21, 'units_1': 11, 'units_2': 6, 'units_3': 11, 'units_4': 11, 'tuner/epochs': 3, 'tuner/initial_epoch': 0, 'tuner/bracket': 2, 'tuner/round': 0}
{'activation': 'relu', 'first_units': 6, 'num_layers': 2, 'units_0': 16, 'units_1': 16, 'units_2': 26, 'units_3': 6, 'units_4': 16, 'tuner/epochs': 20, 'tuner/initial_epoch': 0, 'tuner/bracket': 0, 'tuner/round': 0}


In [20]:
print(X_train.shape)
print(X_test.shape)

(50985, 32)
(16996, 32)


In [21]:
print(y_train.shape)
print(y_test.shape)

(50985,)
(16996,)


In [22]:
# # Evaluate the top 3 models against the test dataset
# top_model = tuner.get_best_models(3)
# for model in top_model:
#     model_loss, model_accuracy = model.evaluate(X_test,y_test,verbose=2)
#     print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")

In [23]:
# Get second best model hyperparameters
second_hyper = tuner.get_best_hyperparameters(2)[1]
second_hyper.values

{'activation': 'relu',
 'first_units': 6,
 'num_layers': 4,
 'units_0': 21,
 'units_1': 11,
 'units_2': 6,
 'units_3': 11,
 'units_4': 11,
 'tuner/epochs': 3,
 'tuner/initial_epoch': 0,
 'tuner/bracket': 2,
 'tuner/round': 0}

In [24]:
# # Compare the performance to the second-best model
# second_model = tuner.get_best_models(2)[1]
# model_loss, model_accuracy = second_model.evaluate(X_test,y_test,verbose=2)
# print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")

## Compile, Train and Evaluate the Model

In [32]:
# Define the model - deep neural net, i.e., the number of input features and hidden nodes for each layer.
number_input_features = len(X_train[0])
hidden_nodes_layer1 = 11 # neural units tried: 2,4,8,16,32,64,128,256
hidden_nodes_layer2 = 26 # multiple layers attempted
hidden_nodes_layer3 = 21
# hidden_nodes_layer4 = 2

nn = tf.keras.models.Sequential()

# First hidden layer
nn.add(
    tf.keras.layers.Dense(units=hidden_nodes_layer1,
                          input_dim=number_input_features,
                          activation="relu")
)

# Additional hidden layers
nn.add(tf.keras.layers.Dense(units=hidden_nodes_layer2,
                             activation="relu"))
nn.add(tf.keras.layers.Dense(units=hidden_nodes_layer3,
                             activation="relu"))
# nn.add(tf.keras.layers.Dense(units=hidden_nodes_layer4,
#                              activation="relu"))

# Output layer
nn.add(tf.keras.layers.Dense(units=1,
                             activation="sigmoid"))

# Check the structure of the model
nn.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_4 (Dense)             (None, 11)                363       
                                                                 
 dense_5 (Dense)             (None, 26)                312       
                                                                 
 dense_6 (Dense)             (None, 21)                567       
                                                                 
 dense_7 (Dense)             (None, 1)                 22        
                                                                 
Total params: 1264 (4.94 KB)
Trainable params: 1264 (4.94 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [33]:
# Compile the model
nn.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

In [34]:
# Train the model
fit_model = nn.fit(X_train,y_train,epochs=5) # no improved accuracy after 2 epochs

Epoch 1/5
1594/1594 [==============================] - 1s 369us/step - loss: nan - accuracy: 0.8097
Epoch 2/5
1594/1594 [==============================] - 1s 364us/step - loss: nan - accuracy: 0.8100
Epoch 3/5
1594/1594 [==============================] - 1s 362us/step - loss: nan - accuracy: 0.8100
Epoch 4/5
1594/1594 [==============================] - 1s 362us/step - loss: nan - accuracy: 0.8100
Epoch 5/5
1594/1594 [==============================] - 1s 362us/step - loss: nan - accuracy: 0.8100


In [35]:
# Evaluate the model using the test data
model_loss, model_accuracy = nn.evaluate(X_test,y_test,verbose=2)
print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")

532/532 - 0s - loss: nan - accuracy: 0.8076 - 159ms/epoch - 298us/step
Loss: nan, Accuracy: 0.807601809501648


In [36]:
from sklearn.metrics import classification_report

y_pred = nn.predict(X_test)
y_pred_classes = (y_pred > 0.5).astype(int)

print(classification_report(y_test, y_pred_classes))

532/532 [==============================] - 0s 219us/step
              precision    recall  f1-score   support

           0       0.81      1.00      0.89     13726
           1       0.00      0.00      0.00      3270

    accuracy                           0.81     16996
   macro avg       0.40      0.50      0.45     16996
weighted avg       0.65      0.81      0.72     16996



/opt/anaconda3/envs/dev/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/dev/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/envs/dev/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [31]:
from sklearn.utils import class_weight
import numpy as np

# Assuming y_train is your target variable
# Calculate class weights
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)

# Convert class weights to a dictionary
class_weight_dict = dict(enumerate(class_weights))

# Compile the model with class weights
nn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Fit the model with class weights
nn.fit(X_train, y_train, epochs=10, batch_size=32, class_weight=class_weight_dict)

Epoch 1/10
1594/1594 [==============================] - 1s 397us/step - loss: nan - accuracy: 0.8100
Epoch 2/10
1594/1594 [==============================] - 1s 393us/step - loss: nan - accuracy: 0.8100
Epoch 3/10
1594/1594 [==============================] - 1s 398us/step - loss: nan - accuracy: 0.8100
Epoch 4/10
1594/1594 [==============================] - 1s 472us/step - loss: nan - accuracy: 0.8100
Epoch 5/10
1594/1594 [==============================] - 1s 390us/step - loss: nan - accuracy: 0.8100
Epoch 6/10
1594/1594 [==============================] - 1s 390us/step - loss: nan - accuracy: 0.8100
Epoch 7/10
1594/1594 [==============================] - 1s 391us/step - loss: nan - accuracy: 0.8100
Epoch 8/10
1594/1594 [==============================] - 1s 393us/step - loss: nan - accuracy: 0.8100
Epoch 9/10
1594/1594 [==============================] - 1s 393us/step - loss: nan - accuracy: 0.8100
Epoch 10/10
1594/1594 [==============================] - 1s 390us/step - loss: nan - accura

In [ ]:
# Export our model to HDF5 file
nn.save('case_leads_to_arrest.h5')

/opt/anaconda3/envs/dev/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
